# Notebook 05 — Leakage-Safe Temporal Feature Engineering
**Component 03: Hybrid Explainable Contamination Risk Prediction Framework**

### Research Purpose
Constructs multi-scale temporal rolling windows (1h, 3h, 6h, 24h rainfall accumulation), lagged water quality indicators, derivative change rates (Δturbidity, ΔpH, ΔTDS), and cyclical seasonal harmonics. Strict spatial and temporal boundaries ensure no cross-site bleed and **zero future information leakage**.

### 🔗 Linked Source Modules & Core Implementations
| Source Module | Core Classes / Functions | Primary Responsibility |
| :--- | :--- | :--- |
| [`src/feature_engineering.py`](../src/feature_engineering.py) | `EnvironmentalFeatureEngineer`, `engineer_features()` | Implements rolling window sums/means, backwards lags, derivatives, and calendar harmonics partitioned strictly by location. |
| [`src/data_loader.py`](../src/data_loader.py) | `load_raw_data()` | Ingests unfeatured multi-station observations. |
| [`src/utils.py`](../src/utils.py) | `get_project_root()`, `load_feature_config()` | Provides sampling interval steps and window configuration. |


In [1]:
# =========================================================================
# 1. ENVIRONMENT SETUP & LINKED FEATURE ENGINEER IMPORT
# Linked Module: src/feature_engineering.py (EnvironmentalFeatureEngineer)
# =========================================================================
import sys
import json
from pathlib import Path
import pandas as pd

cur = Path.cwd().resolve()
root_dir = cur if (cur / 'config' / 'project_config.json').exists() else cur.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.feature_engineering import EnvironmentalFeatureEngineer, engineer_features

try:
    # Load pre-engineered features from processed cache if existing
    feat_file = root_dir / 'data' / 'processed' / 'engineered_features.csv'
    schema_file = root_dir / 'models' / 'metadata' / 'feature_schema.json'
    df_features = pd.read_csv(feat_file)
    with open(schema_file, 'r') as f:
        feature_names = json.load(f)['feature_names']
    print('Loaded pre-engineered features from disk!')
except Exception:
    # Otherwise execute feature engineering pipeline from raw data
    df_features, feature_names = engineer_features()

print(f'Engineered {len(feature_names)} features across {len(df_features):,} records.')


Loaded pre-engineered features from disk!
Engineered 43 features across 140,160 records.


In [2]:
# =========================================================================
# 2. INSPECT TEMPORAL LAG & ROLLING RAINFALL FEATURES
# =========================================================================
key_sample = ['timestamp', 'location_id', 'rain_1h_mm', 'rain_3h_mm', 'rain_24h_mm', 'turbidity_lag_1h', 'turbidity_change_rate', 'is_monsoon', 'future_contamination_event']
df_features[key_sample].head(10)


,timestamp,location_id,rain_1h_mm,rain_3h_mm,rain_24h_mm,turbidity_lag_1h,turbidity_change_rate,is_monsoon,future_contamination_event
0,2025-01-01 00:00:00+05:30,SITE_001,3.94,3.94,3.94,24.83,51.35,1,0
1,2025-01-01 00:00:00+05:30,SITE_002,3.94,3.94,3.94,21.98,60.73,1,0
2,2025-01-01 00:00:00+05:30,SITE_003,3.94,3.94,3.94,23.14,53.22,1,0
3,2025-01-01 00:00:00+05:30,SITE_004,3.94,3.94,3.94,23.09,57.43,1,0
4,2025-01-01 00:15:00+05:30,SITE_001,7.16,7.16,7.16,24.83,51.35,1,0
5,2025-01-01 00:15:00+05:30,SITE_002,7.16,7.16,7.16,21.98,60.73,1,0
6,2025-01-01 00:15:00+05:30,SITE_003,7.16,7.16,7.16,23.14,53.22,1,0
7,2025-01-01 00:15:00+05:30,SITE_004,7.16,7.16,7.16,23.09,57.43,1,0
8,2025-01-01 00:30:00+05:30,SITE_001,10.19,10.19,10.19,24.83,51.35,1,0
9,2025-01-01 00:30:00+05:30,SITE_002,10.19,10.19,10.19,21.98,60.73,1,0


### Verification of Leakage Safety
- Rolling sums and means strictly use trailing backwards windows (`shift` and `rolling`).
- Target variables predict $t+1$ to $t+12$ (15 min to 3 hours in the future) strictly after the feature cutoff $t$.
